In [ ]:
import os
from typing import Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler


class LearningPathRecommender:
    """
    Learning Path Recommendation System
    using Collaborative Filtering (Matrix Factorization)
    """

    def __init__(
        self,
        n_components: int = 20,
        random_state: int = 42,
    ) -> None:

        self.n_components = n_components
        self.random_state = random_state

        # datasets
        self.interactions_df = None
        self.courses_df = None

        # indexes
        self.intern_index_ = None
        self.course_index_ = None

        # model
        self.svd = None
        self.intern_factors_ = None
        self.course_factors_ = None

    def load_data(
        self,
        interactions_path="intern_ratings.csv",
        courses_path="course_metadata.csv",
    ):

        # check files exist
        if not os.path.exists(interactions_path):
            raise FileNotFoundError(
                f"{interactions_path} not found"
            )

        if not os.path.exists(courses_path):
            raise FileNotFoundError(
                f"{courses_path} not found"
            )

        # load datasets
        self.interactions_df = pd.read_csv(interactions_path)

        self.courses_df = pd.read_csv(courses_path)

        # validate columns
        required_inter_cols = {
            "intern_id",
            "course_id",
            "rating"
        }

        required_course_cols = {
            "course_id",
            "title"
        }

        if not required_inter_cols.issubset(
            self.interactions_df.columns
        ):
            raise ValueError(
                "Interactions dataset missing required columns"
            )

        if not required_course_cols.issubset(
            self.courses_df.columns
        ):
            raise ValueError(
                "Courses dataset missing required columns"
            )

        print("Datasets loaded successfully!")

    def _build_interaction_matrix(self):

        pivot_df = self.interactions_df.pivot_table(
            index="intern_id",
            columns="course_id",
            values="rating",
            aggfunc="mean"
        )

        # save indexes
        self.intern_index_ = pivot_df.index

        self.course_index_ = pivot_df.columns

        # replace NaN with 0
        interaction_matrix = pivot_df.fillna(0.0).to_numpy()

        return pivot_df, interaction_matrix

    def fit(self):

        _, interaction_matrix = self._build_interaction_matrix()

        n_components = min(
            self.n_components,
            min(interaction_matrix.shape) - 1
        )

        self.svd = TruncatedSVD(
            n_components=n_components,
            random_state=self.random_state
        )

        # learn intern latent factors
        self.intern_factors_ = self.svd.fit_transform(
            interaction_matrix
        )

        # learn course latent factors
        self.course_factors_ = self.svd.components_.T

        print("Model trained successfully!")

    def _predict_scores_for_all_courses(self):

        score_matrix = np.dot(
            self.intern_factors_,
            self.course_factors_.T
        )

        scaler = MinMaxScaler()

        score_matrix_scaled = scaler.fit_transform(
            score_matrix
        )

        return score_matrix_scaled

    def recommend_for_intern(
        self,
        intern_id,
        top_n=5,
        filter_by_difficulty=None,
        preferred_category=None,
    ):

        if intern_id not in self.intern_index_:
            raise ValueError(
                f"Intern {intern_id} not found"
            )

        intern_pos = self.intern_index_.get_loc(
            intern_id
        )

        score_matrix = self._predict_scores_for_all_courses()

        intern_scores = score_matrix[intern_pos]

        # already rated courses
        seen_courses = self.interactions_df.loc[
            self.interactions_df["intern_id"] == intern_id,
            "course_id"
        ].unique()

        recommendations = []

        for col_idx, course_id in enumerate(
            self.course_index_
        ):

            if course_id in seen_courses:
                continue

            score = intern_scores[col_idx]

            course_row = self.courses_df[
                self.courses_df["course_id"] == course_id
            ]

            if course_row.empty:
                continue

            course_row = course_row.iloc[0]

            rec = {
                "intern_id": intern_id,
                "course_id": course_id,
                "predicted_score": float(score),
                "title": course_row.get("title", ""),
                "category": course_row.get("category", ""),
                "difficulty": course_row.get("difficulty", ""),
                "duration_hours": course_row.get(
                    "duration_hours",
                    np.nan
                ),
            }

            recommendations.append(rec)

        rec_df = pd.DataFrame(recommendations)

        if filter_by_difficulty:
            rec_df = rec_df[
                rec_df["difficulty"].str.lower()
                == filter_by_difficulty.lower()
            ]

        if preferred_category:
            rec_df = rec_df[
                rec_df["category"].str.lower()
                == preferred_category.lower()
            ]

        rec_df = rec_df.sort_values(
            by="predicted_score",
            ascending=False
        )

        return rec_df.head(top_n).reset_index(drop=True)

    def build_learning_path(
        self,
        intern_id,
        top_n=10,
        strategy="difficulty_ascending",
    ):

        rec_df = self.recommend_for_intern(
            intern_id=intern_id,
            top_n=top_n * 3
        )

        if strategy == "difficulty_ascending":

            difficulty_order = {
                "beginner": 0,
                "intermediate": 1,
                "advanced": 2
            }

            rec_df["difficulty_rank"] = rec_df[
                "difficulty"
            ].apply(
                lambda x: difficulty_order.get(
                    str(x).lower(),
                    1
                )
            )

            rec_df = rec_df.sort_values(
                by=["difficulty_rank", "predicted_score"],
                ascending=[True, False]
            )

            rec_df = rec_df.drop(
                columns=["difficulty_rank"]
            )

        elif strategy == "shortest_first":

            rec_df = rec_df.sort_values(
                by=["duration_hours", "predicted_score"],
                ascending=[True, False]
            )

        return rec_df.head(top_n).reset_index(drop=True)

In [ ]:
recommender = LearningPathRecommender(
    n_components=20,
    random_state=42
)

In [ ]:
recommender.load_data(
    interactions_path="intern_ratings.csv",
    courses_path="course_metadata.csv",
)

Datasets loaded successfully!


In [ ]:
recommender.fit()

Model trained successfully!


In [ ]:
recommendations = recommender.recommend_for_intern(
    intern_id=10,
    top_n=5
)

recommendations

,intern_id,course_id,predicted_score,title,category,difficulty,duration_hours
0,10,C027,0.447043,Power BI,Data Science,Intermediate,25
1,10,C029,0.421313,Excel Analytics,Data Science,Beginner,15
2,10,C028,0.386898,Data Visualization,Data Science,Intermediate,30
3,10,C003,0.374638,Deep Learning,AI/ML,Advanced,50
4,10,C011,0.367806,UI Design Basics,UI/UX,Beginner,20


In [ ]:
learning_path = recommender.build_learning_path(
    intern_id=10,
    top_n=5,
    strategy="difficulty_ascending"
)

learning_path

,intern_id,course_id,predicted_score,title,category,difficulty,duration_hours
0,10,C029,0.421313,Excel Analytics,Data Science,Beginner,15
1,10,C011,0.367806,UI Design Basics,UI/UX,Beginner,20
2,10,C016,0.334600,Cyber Security Basics,Cybersecurity,Beginner,25
3,10,C022,0.273515,Azure Fundamentals,Cloud,Beginner,30
4,10,C027,0.447043,Power BI,Data Science,Intermediate,25


In [ ]:
recommender.recommend_for_intern(
    intern_id=10,
    top_n=5,
    filter_by_difficulty="Beginner"
)

,intern_id,course_id,predicted_score,title,category,difficulty,duration_hours
0,10,C029,0.421313,Excel Analytics,Data Science,Beginner,15
1,10,C011,0.367806,UI Design Basics,UI/UX,Beginner,20
2,10,C016,0.334600,Cyber Security Basics,Cybersecurity,Beginner,25
3,10,C022,0.273515,Azure Fundamentals,Cloud,Beginner,30
4,10,C001,0.149522,Python Basics,AI/ML,Beginner,20


In [ ]:
recommender.recommend_for_intern(
    intern_id=10,
    top_n=5,
    preferred_category="AI/ML"
)

,intern_id,course_id,predicted_score,title,category,difficulty,duration_hours
0,10,C003,0.374638,Deep Learning,AI/ML,Advanced,50
1,10,C004,0.354615,Data Science,AI/ML,Intermediate,35
2,10,C005,0.208205,Computer Vision,AI/ML,Advanced,45
3,10,C002,0.203352,Machine Learning,AI/ML,Intermediate,40
4,10,C001,0.149522,Python Basics,AI/ML,Beginner,20
